In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

In [2]:
X_train = pd.read_csv(r'D:\home loan credit risk\artifact\data\final\X_train_final.csv')
X_test = pd.read_csv(r'D:\home loan credit risk\artifact\data\final\X_test_final.csv')

y_train = pd.read_csv(r'D:\home loan credit risk\artifact\data\data_splits\y_train.csv')
y_test = pd.read_csv(r'D:\home loan credit risk\artifact\data\data_splits\y_test.csv')

In [3]:
from src.utils.main_utils import downcast_df_variables
X_train = downcast_df_variables(X_train)
X_test = downcast_df_variables(X_test)

# STEPWISE LOGISTIC REGRESSION FEATURE SELECTION

In [ ]:

import pandas as pd
import statsmodels.api as sm
import warnings

warnings.filterwarnings("ignore")


def stepwise_logistic_regression(
        X,
        y,
        entry_threshold=0.05,
        exit_threshold=0.10,
        verbose=True):

    """
    TRUE Stepwise Logistic Regression

    Parameters:
    X : DataFrame
        Feature matrix

    y : Series
        Target variable (0/1)

    entry_threshold : float
        p-value to ENTER model

    exit_threshold : float
        p-value to REMOVE variable

    verbose : bool
        Print steps

    Returns:
    selected_features : list
    final_model : fitted model
    """

    selected_features = []
    remaining_features = list(X.columns)

    while True:

        changed = False

        # -------------------------
        # FORWARD STEP
        # -------------------------

        pvals_forward = pd.Series(
            index=remaining_features,
            dtype=float
        )

        for feature in remaining_features:

            try:

                test_features = selected_features + [feature]

                X_model = sm.add_constant(
                    X[test_features],
                    has_constant='add'
                )

                model = sm.Logit(y, X_model)

                result = model.fit(disp=0,
                                method='lbfgs',
                                maxiter=200)

                pvals_forward[feature] = \
                    result.pvalues[feature]

            except:

                pvals_forward[feature] = 1

        if not pvals_forward.empty:

            best_feature = pvals_forward.idxmin()
            best_pval = pvals_forward.min()

            if best_pval < entry_threshold:

                selected_features.append(best_feature)
                remaining_features.remove(best_feature)

                changed = True

                if verbose:
                    print(
                        f"Add: {best_feature} "
                        f"(p={best_pval:.6f})"
                    )

        # -------------------------
        # BACKWARD STEP
        # -------------------------

        if selected_features:

            try:

                X_model = sm.add_constant(
                    X[selected_features],
                    has_constant='add'
                )

                model = sm.Logit(y, X_model)

                result = model.fit(disp=0)

                pvals_backward = \
                    result.pvalues.iloc[1:]

                worst_pval = \
                    pvals_backward.max()

                if worst_pval > exit_threshold:

                    worst_feature = \
                        pvals_backward.idxmax()

                    selected_features.remove(
                        worst_feature
                    )

                    remaining_features.append(
                        worst_feature
                    )

                    changed = True

                    if verbose:
                        print(
                            f"Remove: {worst_feature} "
                            f"(p={worst_pval:.6f})"
                        )

            except:
                pass

        if not changed:
            break

    # -------------------------
    # FINAL MODEL
    # -------------------------

    X_final = sm.add_constant(
        X[selected_features],
        has_constant='add'
    )

    final_model = sm.Logit(
        y,
        X_final
    ).fit()

    return selected_features, final_model

selected_features, final_model = stepwise_logistic_regression(
        X_train,
        y_train
    )
    
print("\nSelected Features:")
print(selected_features)

In [4]:
selected_features = ['EXT_SOURCE_3', 'ANNUITY_CREDIT_RATIO', 'EXT_SOURCE_WEIGHTED', 'IP_WORST_DPD_720D', 'GOODS_CREDIT_RATIO', 'OCCUPATION_GROUP', 'CB_AVG_ATM_WITHDRAWAL_FREQ_6M', 'PA_RATIO_APPROVED_LOANS', 'ANNUITY_TO_AGE_RATIO', 'PA_AVG_AMT_ANNUITY_POS', 'PA_RATIO_CREDIT_APPLICATION_POS', 'B_DEBT_TO_CREDIT_RATIO', 'IP_RATIO_LATE_PAYMENTS_2160D', 'YEARS_EMPLOYED', 'EXT_SOURCE_1', 'AMT_GOODS_PRICE', 'REGION_RATING_CLIENT_W_CITY', 'NAME_EDUCATION_TYPE', 'ORG_GROUP', 'B_NUM_ACTIVE_CREDIT_720D', 'B_CREDIT_DURATION_MIN', 'IP_NUM_COMPLETED_LOANS', 'PA_AVG_RISK_WEIGHT_1080D', 'CODE_GENDER', 'CREDIT_EXT_SOURCE_PRODUCT', 'ID_TO_AGE_RATIO', 'PA_RATIO_CREDIT_APPLICATION_Cash', 'IP_WORST_DPD_90D', 'FLAG_DOCUMENT_3', 'EXT_SOURCE_X_INCOME', 'CB_WT_CREDIT_UTIL_TREND_3M_12M', 'PA_RATIO_HIGH_RISK_CHANNEL', 'B_OVERDUE_TO_CREDIT_RATIO', 'B_AMT_CREDIT_SUM_DEBT_SUM', 'BUREAU_DAYS_CREDIT_ENDDATE_MAX', 'IP_RATIO_AMT_PAID_OWED_360D', 'PA_AVG_RISK_WEIGHT_360D', 'EXT_SOURCE_RANGE', 'FLOORSMAX_MEDI', 'NAME_FAMILY_STATUS', 'AMT_CREDIT', 'YEARS_AGE', 'IP_MIN_RATIO_PAYMENT_INSTALMENT', 'WORST_DPD_DEF_POS_CASH_72M', 'B_MIN_REPAYMENT_DAYS_DIFF', 'REG_CITY_NOT_LIVE_CITY', 'IP_RATIO_EARLY_PAYMENTS_1080D', 'B_DAYS_CREDIT_MAX', 'CREDIT_GOODS_DIFF_AMT', 'YEARS_REGISTRATION', 'PCB_HISTORY_LENGTH', 'B_CLOSED_CREDIT_COUNT', 'BUREAU_DAYS_CREDIT_ENDDATE_MEAN', 'IP_NUM_UNDERPAID_INSTALLMENTS_720D', 'IP_MEAN_INSTALMENT_PAYMENT_DIFF', 'NAME_INCOME_TYPE', 'B_NUM_ACTIVE_CREDIT_270D', 'B_MAX_REPAYMENT_DAYS_DIFF', 'MONTHS_BALANCE_SUM', 'YEARS_ID_PUBLISH', 'EMPLOYMENT_GAP', 'PA_MAX_DOWN_PAYMENT_RATE', 'REG_CITY_NOT_WORK_CITY', 'B_DAYS_CREDIT_MEAN', 'PA_MAX_AMT_GOODS_PRICE', 'PA_RATIO_HIGH_RISK_YIELD_LOANS_720D', 'CREDIT_PER_PERSON']
len(selected_features)

67

# metrics UTIL

In [5]:

import numpy as np
from sklearn.metrics import roc_auc_score, roc_curve

def auc_gini_ks(y_true, y_pred_prob):
    """
    Calculate AUC, Gini, and KS statistic for credit risk models.

    Parameters:
    y_true : array-like, true binary labels (0 = good, 1 = bad)
    y_pred_prob : array-like, predicted probabilities of default (1 = bad)

    Returns:
    dict with keys: 'AUC', 'Gini', 'KS'
    """
    # AUC
    auc = roc_auc_score(y_true, y_pred_prob)
    
    # Gini coefficient
    gini = 2 * auc - 1
    
    # KS statistic
    fpr, tpr, thresholds = roc_curve(y_true, y_pred_prob)
    ks = max(tpr - fpr)
    
    print("AUC", auc)
    print("Gini", gini)
    print("KS", ks)

# LOGISTIC REGRESSION

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

X_train = X_train[selected_features]
X_test  = X_test[selected_features]



final_model = LogisticRegression(
    penalty='l1',          # removes weak features
    solver='liblinear',    # required for L1
    C=0.1,                 # strong regularization
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)

final_model.fit(X_train, y_train)
threshold = 0.5

# train evaluate 
y_train_pred_prob = final_model.predict_proba(X_train)
y_train_pred_prob = y_train_pred_prob[:,1]
y_train_pred =  (y_train_pred_prob>threshold).astype(int)
print(classification_report(y_train, y_train_pred))

#  test evaluate
y_pred_prob = final_model.predict_proba(X_test)
y_pred_prob = y_pred_prob[:,1]
y_pred =  (y_pred_prob>threshold).astype(int)
print(classification_report(y_test, y_pred))

c:\ProgramData\anaconda3\envs\homecredit\Lib\site-packages\sklearn\utils\validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


              precision    recall  f1-score   support

           0       0.96      0.71      0.82    226148
           1       0.17      0.70      0.28     19860

    accuracy                           0.71    246008
   macro avg       0.57      0.70      0.55    246008
weighted avg       0.90      0.71      0.77    246008

              precision    recall  f1-score   support

           0       0.96      0.71      0.82     56538
           1       0.17      0.70      0.28      4965

    accuracy                           0.71     61503
   macro avg       0.57      0.70      0.55     61503
weighted avg       0.90      0.71      0.77     61503



In [7]:
auc_gini_ks(y_train, y_train_pred_prob)

AUC 0.7736687391270884
Gini 0.5473374782541769
KS 0.40992376442123896


In [8]:
#  test evaluate
auc_gini_ks(y_test, y_pred_prob)


AUC 0.7723155975588716
Gini 0.5446311951177432
KS 0.41081973688471324


In [9]:

# Get coefficients
coef = final_model.coef_[0]  # shape: (n_features,)

# Put into DataFrame
importance = pd.DataFrame({"feature": selected_features, "coefficient": coef})
importance["abs_coef"] = importance["coefficient"].abs()
importance = importance.sort_values(by="abs_coef", ascending=False)
importance.head(20)


,feature,coefficient,abs_coef
2,EXT_SOURCE_WEIGHTED,-0.686748,0.686748
9,PA_AVG_AMT_ANNUITY_POS,-0.551379,0.551379
15,AMT_GOODS_PRICE,-0.551207,0.551207
23,CODE_GENDER,-0.544296,0.544296
6,CB_AVG_ATM_WITHDRAWAL_FREQ_6M,-0.510232,0.510232
20,B_CREDIT_DURATION_MIN,-0.506593,0.506593
8,ANNUITY_TO_AGE_RATIO,-0.478303,0.478303
31,PA_RATIO_HIGH_RISK_CHANNEL,-0.468775,0.468775
32,B_OVERDUE_TO_CREDIT_RATIO,-0.456846,0.456846
13,YEARS_EMPLOYED,-0.453660,0.453660


In [10]:
importance['LOG_REG_TOP_30'] = 0
filt_top_30 = importance.index <= 30
importance.loc[filt_top_30,'LOG_REG_TOP_30'] = 1


# FEATURE SELECTION METHODS 

## 1) CORR REMOVED  (0.6)  LOGistic  REGression

* used the feature selected by the stepwise log reg and then again tried increased threshold of corr ! 0.6 to perform aggressive corr filtring
* got 58 corr_selected_features that are kept after corr filter
* got good score

In [11]:
from src.components.module_05_woe_binning_and_selection import WOEFeatureSelector
wfe = WOEFeatureSelector(corr_threshold=0.6)
iv_df = pd.read_csv(r'D:\home loan credit risk\artifact\binning\prebin\iv_df.csv')
corr_selected_features = wfe.correlation_filter(X_train,selected_features,iv_df)

2026-04-12 08:03:59,247 - module_05_feature_engineering.py - INFO - Total features before : 67
2026-04-12 08:03:59,249 - module_05_feature_engineering.py - INFO - Total features after  : 58
2026-04-12 08:03:59,254 - module_05_feature_engineering.py - INFO - Dropped features      : 9
2026-04-12 08:03:59,256 - module_05_feature_engineering.py - INFO - Removed 9 features due to high correlation (> 0.6)
2026-04-12 08:03:59,261 - module_05_feature_engineering.py - INFO - Remaining feature list final shape is  58


In [12]:
len(corr_selected_features)

58

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

X_train = X_train[corr_selected_features]
X_test  = X_test[corr_selected_features]



final_model = LogisticRegression(
    penalty='l1',          # removes weak features
    solver='liblinear',    # required for L1
    C=0.1,                 # strong regularization
    class_weight='balanced',
    max_iter=2000,
    random_state=42
)

final_model.fit(X_train, y_train)
threshold = 0.5

# train evaluate 
y_train_pred_prob = final_model.predict_proba(X_train)
y_train_pred_prob = y_train_pred_prob[:,1]
y_train_pred =  (y_train_pred_prob>threshold).astype(int)
print(classification_report(y_train, y_train_pred))

#  test evaluate
y_pred_prob = final_model.predict_proba(X_test)
y_pred_prob = y_pred_prob[:,1]
y_pred =  (y_pred_prob>threshold).astype(int)
print(classification_report(y_test, y_pred))

c:\ProgramData\anaconda3\envs\homecredit\Lib\site-packages\sklearn\utils\validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


              precision    recall  f1-score   support

           0       0.96      0.70      0.81    226148
           1       0.17      0.70      0.28     19860

    accuracy                           0.70    246008
   macro avg       0.57      0.70      0.55    246008
weighted avg       0.90      0.70      0.77    246008

              precision    recall  f1-score   support

           0       0.96      0.71      0.81     56538
           1       0.17      0.70      0.28      4965

    accuracy                           0.71     61503
   macro avg       0.57      0.70      0.55     61503
weighted avg       0.90      0.71      0.77     61503



In [14]:
auc_gini_ks(y_train, y_train_pred_prob)

AUC 0.7725085147298401
Gini 0.5450170294596801
KS 0.4088941701520278


In [15]:
auc_gini_ks(y_test, y_pred_prob)

AUC 0.7711209924421605
Gini 0.542241984884321
KS 0.4096303221564001


In [16]:
coef = final_model.coef_[0]  # shape: (n_features,)


# Put into DataFrame
importance_1 = pd.DataFrame({"feature": final_model.feature_names_in_, "coefficient": coef})
importance_1["abs_coef"] = importance_1["coefficient"].abs()
importance_1 = importance_1.sort_values(by="abs_coef", ascending=False)
importance_1.head(20)


,feature,coefficient,abs_coef
2,EXT_SOURCE_WEIGHTED,-0.732045,0.732045
9,PA_AVG_AMT_ANNUITY_POS,-0.543353,0.543353
20,B_CREDIT_DURATION_MIN,-0.541044,0.541044
6,CB_AVG_ATM_WITHDRAWAL_FREQ_6M,-0.518902,0.518902
23,CODE_GENDER,-0.490237,0.490237
17,NAME_EDUCATION_TYPE,-0.488340,0.488340
30,PA_RATIO_HIGH_RISK_CHANNEL,-0.487428,0.487428
15,AMT_GOODS_PRICE,-0.454059,0.454059
13,YEARS_EMPLOYED,-0.447014,0.447014
31,B_OVERDUE_TO_CREDIT_RATIO,-0.435399,0.435399


In [17]:
importance_1['LOG_REG_CORR_TOP_30'] = 0
filt_top_30 = importance_1.index <= 30
importance_1.loc[filt_top_30,'LOG_REG_CORR_TOP_30'] = 1

## 2) RECURSIVE FEATURE ELIMINATION WITH LOG REG





* RFE: Used when the number of features to select (n_features_to_select) is known in advance.
* RFECV: Automatically tunes the number of features by using cross-validation to find the subset that yields the best model performance.

* used the dataset with removed corr feature ! rem feature = 58
* then performed the RFE on them n_features_to_select = 25 as threshold

In [18]:
from sklearn.feature_selection import RFE


final_model = LogisticRegression(
        penalty='l2',          # removes weak features
        solver='liblinear',    # required for L1
        C=1,                 # strong regularization
        class_weight='balanced',
        max_iter=2000,
        random_state=42
    )

X_train = X_train[corr_selected_features]
X_test = X_test[corr_selected_features]


rfe = RFE(final_model,n_features_to_select=30)
rfe.fit(X_train,y_train.values.ravel())

train_prob = rfe.predict_proba(X_train)
test_prop = rfe.predict_proba(X_test)

In [19]:
train_prob = train_prob[:,1]
test_prop = test_prop[:,1]

In [20]:
auc_gini_ks(y_test, test_prop)

AUC 0.7661226412899779
Gini 0.5322452825799557
KS 0.3985139209102367


In [21]:
auc_gini_ks(y_train, train_prob)

AUC 0.767173878468415
Gini 0.53434775693683
KS 0.4009487508478839


In [22]:
importance_2 = pd.DataFrame({
    'feature':rfe.get_feature_names_out(),
    'coef':rfe.estimator_.coef_[0],
})

importance_2 = importance_2.sort_values(by='coef',ascending=False)
importance_2 = importance_2.reset_index(drop=True)

In [23]:
importance_2['RFE_TOP_30'] = 0
filt_top_30 = importance_2.index <= 30
importance_2.loc[filt_top_30,'RFE_TOP_30'] = 1

## 3) LASSO REGULARIZATION LOG REG

* tried diff diff c values for lasso regularization l1 that eliminates if features
* select the best c values and trian the proper log reg and saved the metrics

In [24]:
X_train.shape

(246008, 58)

In [25]:
import numpy as np
from sklearn.metrics import roc_auc_score


# See exactly how many features each C value keeps
c_values = [0.0001, 0.0003, 0.0005, 0.0008,
            0.001,  0.002,  0.003,  0.005,
            0.008,  0.01,   0.015,  0.02,
            0.03,   0.05,   0.08,   0.1]
c_scores_dict = {}

for c in c_values:
    m = LogisticRegression(
        penalty='l1', solver='liblinear',
        C=c, class_weight='balanced',
        max_iter=2000, random_state=42
    )
    m.fit(X_train, y_train.values.ravel())
    pred_train = m.predict_proba(X_train)
    pred_train = pred_train[:,1]
    auc_score = roc_auc_score(y_train,pred_train)    
    c_scores_dict[c] = auc_score
    n = (m.coef_[0] != 0).sum()
    print(f"C={c:.4f} → {n} features kept, AUC_train:{auc_score}" )
    

C=0.0001 → 1 features kept, AUC_train:0.7158367830923081
C=0.0003 → 5 features kept, AUC_train:0.7267306357281984
C=0.0005 → 12 features kept, AUC_train:0.7399928955524869
C=0.0008 → 21 features kept, AUC_train:0.7502127781829759
C=0.0010 → 26 features kept, AUC_train:0.7553184652170408
C=0.0020 → 37 features kept, AUC_train:0.7654009024310667
C=0.0030 → 47 features kept, AUC_train:0.7682538895515331
C=0.0050 → 52 features kept, AUC_train:0.7704274654794325
C=0.0080 → 54 features kept, AUC_train:0.771561856149564
C=0.0100 → 55 features kept, AUC_train:0.7718658639466128
C=0.0150 → 58 features kept, AUC_train:0.7721973321715492
C=0.0200 → 58 features kept, AUC_train:0.77233377086374
C=0.0300 → 58 features kept, AUC_train:0.7724336646744236
C=0.0500 → 58 features kept, AUC_train:0.7724854474627664
C=0.0800 → 58 features kept, AUC_train:0.7725036462054694
C=0.1000 → 58 features kept, AUC_train:0.7725085147298401


In [26]:
# SELECTEF THE C VALUE AND RETRAIN
final_model = LogisticRegression(
        penalty='l1',          # removes weak features
        solver='liblinear',    # required for L1
        C=0.001,                 # strong regularization
        class_weight='balanced',
        max_iter=2000,
        random_state=42
    )

final_model.fit(X_train,y_train)
train_prob = final_model.predict_proba(X_train)
train_prob = train_prob[:,1]

test_prob = final_model.predict_proba(X_test)
test_prob = test_prob[:,1]



c:\ProgramData\anaconda3\envs\homecredit\Lib\site-packages\sklearn\utils\validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [27]:
auc_gini_ks(y_train,train_prob)

AUC 0.7553184652170408
Gini 0.5106369304340816
KS 0.38073732018143314


In [28]:
auc_gini_ks(y_test,test_prob)


AUC 0.7562542630562225
Gini 0.5125085261124449
KS 0.386573690672872


In [29]:
importance_3 = pd.DataFrame({
    'feature': final_model.feature_names_in_,
    'coeff': final_model.coef_[0]  
})
importance_3 = importance_3.reset_index(drop=True)

In [30]:
importance_3['LASSO_SELECTED_FEATURES'] = 0
filt_coef = importance_3['coeff'] !=0
importance_3.loc[filt_coef,'LASSO_SELECTED_FEATURES'] = 1

In [31]:
importance_3 = importance_3.sort_values(by='LASSO_SELECTED_FEATURES',ascending=False)

In [32]:
importance_3[importance_3['LASSO_SELECTED_FEATURES']!=0]

,feature,coeff,LASSO_SELECTED_FEATURES
0,EXT_SOURCE_3,-0.123459,1
1,ANNUITY_CREDIT_RATIO,-0.413594,1
2,EXT_SOURCE_WEIGHTED,-0.772074,1
3,IP_WORST_DPD_720D,-0.197485,1
4,GOODS_CREDIT_RATIO,-0.306316,1
5,OCCUPATION_GROUP,-0.249578,1
6,CB_AVG_ATM_WITHDRAWAL_FREQ_6M,-0.216319,1
7,PA_RATIO_APPROVED_LOANS,-0.110001,1
8,ANNUITY_TO_AGE_RATIO,-0.033065,1
9,PA_AVG_AMT_ANNUITY_POS,-0.135504,1


## 4) STABILITIY SELECTION METHOD

* NOT USEFUL FOR WOE BINNED DATA THE NOISE IS NOT THERE IN DATA
* it finds the feature that are consistenly important across  many random sample of data!
* Stability Selection is like  Select features that stay important again and again across 
many random samples of data.

In [33]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

n_iterations = 100
feature_counts = pd.Series(0, index=X_train.columns)

for i in range(n_iterations):
    
    # Step 1 — Random 80% subsample each iteration
    idx = np.random.choice(
        len(X_train),
        size=int(0.8 * (X_train.shape[0])),
        replace=False
    )
    X_sub = X_train.iloc[idx]
    y_sub = y_train.iloc[idx].values.ravel()

    # Step 2 — Lasso on subsample
    m = LogisticRegression(
        penalty='l2',
        solver='liblinear',
        C=1,
        class_weight='balanced',
        max_iter=2000,
        random_state=i       # ← different seed each time
    )
    m.fit(X_sub, y_sub)

    # Step 3 — Record selected features
    selected = X_train.columns[m.coef_[0] != 0]
    feature_counts[selected] += 1

    if (i+1) % 10 == 0:
        print(f"Iteration {i+1}/100 done")

# Step 4 — Calculate frequency
stability_df = pd.DataFrame({
    'feature': feature_counts.index,
    'selection_frequency': feature_counts.values / n_iterations
}).sort_values('selection_frequency', ascending=False).reset_index(drop=True)


stability_df['STABILITY_SELECTED'] = (
    stability_df['selection_frequency'] >= 0.60
).astype(int)

print(f"\n Stable features (>60%): {stability_df['STABILITY_SELECTED'].sum()}")

Iteration 10/100 done
Iteration 20/100 done
Iteration 30/100 done
Iteration 40/100 done
Iteration 50/100 done
Iteration 60/100 done
Iteration 70/100 done
Iteration 80/100 done
Iteration 90/100 done
Iteration 100/100 done

 Stable features (>60%): 58


In [34]:
importance_4 = stability_df.copy()

In [35]:
stable_feature_selected = stability_df[stability_df['STABILITY_SELECTED']==1]['feature'].tolist()
stable_feature_removed = stability_df[stability_df['STABILITY_SELECTED']==0]['feature'].tolist()

In [36]:
# check score on stable featuers 

final_model = LogisticRegression(
        penalty='l2',          # removes weak features
        solver='liblinear',    # required for L1
        C=1,                 # strong regularization
        class_weight='balanced',
        max_iter=2000,
        random_state=42
    )

final_model.fit(X_train[stable_feature_selected],y_train.values.ravel())
train_prob = final_model.predict_proba(X_train[stable_feature_selected])
train_prob = train_prob[:,1]

test_prob = final_model.predict_proba(X_test[stable_feature_selected])
test_prob = test_prob[:,1]


In [37]:
auc_gini_ks(y_train,train_prob)

AUC 0.7725193381012009
Gini 0.5450386762024018
KS 0.40908061330528833


In [38]:
auc_gini_ks(y_test,test_prob)

AUC 0.7710706951917873
Gini 0.5421413903835746
KS 0.4096723119354317


## 5) PERMUTATION IMPORTANCE

* permutation imporatnce is the feature selection or importance technique that  measure  how important feature is by  randomly shuffling one feature at time and cehcking how the model performance decreases
* If the model performance drops significantly after shuffling, the feature is considered important because the model relied on it for prediction.

In [39]:
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
final_model = LogisticRegression(
        penalty='l2',          # removes weak features
        solver='liblinear',    # required for L1
        C=1,                 # strong regularization
        class_weight='balanced',
        max_iter=2000,
        random_state=42
    )


final_model.fit(X_train, y_train.values.ravel())

perm = permutation_importance(
    final_model, X_test, y_test.values.ravel(),
    n_repeats=10, scoring='roc_auc',
    random_state=42, n_jobs=-1
)


In [40]:
perm_df = pd.DataFrame({
    'feature': final_model.feature_names_in_,
    'perm_importance': perm.importances_mean,
    'perm_std': perm.importances_std
}).sort_values('perm_importance', ascending=False)


In [41]:
# Flag BEST FEATURE
perm_df['PERM_BEST_SELECTED'] = 0
perm_filt = perm_df['perm_importance']> 0 

perm_df.loc[perm_filt, 'PERM_BEST_SELECTED'] = 1
perm_selected_feature = perm_df[perm_df['PERM_BEST_SELECTED'] == 1]['feature'].tolist()
perm_removed_feature = perm_df[perm_df['PERM_BEST_SELECTED'] == 0]['feature'].tolist()

In [42]:
# SELECTEF THE C VALUE AND RETRAIN
final_model = LogisticRegression(
        penalty='l2',          # removes weak features
        solver='liblinear',    # required for L1
        C=1,                 # strong regularization
        class_weight='balanced',
        max_iter=2000,
        random_state=42
    )


final_model.fit( X_train[perm_selected_feature],y_train)
train_prob = final_model.predict_proba( X_train[perm_selected_feature])
train_prob = train_prob[:,1]

test_prob = final_model.predict_proba(X_test[perm_selected_feature])
test_prob = test_prob[:,1]


c:\ProgramData\anaconda3\envs\homecredit\Lib\site-packages\sklearn\utils\validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [43]:
auc_gini_ks(y_train,train_prob)

AUC 0.7724192211924921
Gini 0.5448384423849841
KS 0.40844673704309453


In [44]:
auc_gini_ks(y_test,test_prob)

AUC 0.771263273919595
Gini 0.5425265478391901
KS 0.4107463589710377


## 6) SHAP FEATURE SELECTION

* shap values are mainly derived from the idea of game theory that how each particular feature contributes to the model?

In [45]:
import shap
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

np.random.seed(42) 

# Train on all 58 features 
lr_model = LogisticRegression(
    penalty='l2', solver='lbfgs', C=1.0,
    class_weight='balanced', max_iter=2000, random_state=42
)
lr_model.fit(X_train, y_train.values.ravel())

# 2 Get SHAP values ──
explainer = shap.LinearExplainer(
    lr_model, X_train,
    feature_perturbation='correlation_dependent'
)

X_sample = X_train.sample(70000, random_state=42)
shap_values = explainer.shap_values(X_sample)

#  Step 3: Rank features by SHAP importance ──
shap_df = pd.DataFrame({
    'feature': X_train.columns,
    'shap_importance': np.abs(shap_values).mean(axis=0)
}).sort_values('shap_importance', ascending=False).reset_index(drop=True)

print("SHAP Feature Ranking:")
print(shap_df)

ranked_features = shap_df['feature'].tolist()

# ── Step 4: Progressive AUC tracking ──
# Add features one by one, track AUC at each step
results = []

for n in range(1, len(ranked_features) + 1):
    features_subset = ranked_features[:n]

    m = LogisticRegression(
        penalty='l2', solver='lbfgs', C=1.0,
        class_weight='balanced', max_iter=2000, random_state=42
    )
    m.fit(X_train[features_subset], y_train.values.ravel())

    train_auc = roc_auc_score(y_train, m.predict_proba(X_train[features_subset])[:,1])
    test_auc  = roc_auc_score(y_test,  m.predict_proba(X_test[features_subset])[:,1])

    results.append({
        'n_features': n,
        'feature_added': ranked_features[n-1],
        'train_auc': round(train_auc, 4),
        'test_auc': round(test_auc, 4),
        'gini': round(2*test_auc - 1, 4)
    })

    if n % 5 == 0:
        print(f"n={n:3d} | Test AUC: {test_auc:.4f} | Gini: {2*test_auc-1:.4f}")

results_df = pd.DataFrame(results)

# ── Step 5: Find elbow point ──
# Where does AUC stop meaningfully improving?
max_auc = results_df['test_auc'].max()
threshold = max_auc - 0.005  # within 0.005 of best AUC

elbow = results_df[results_df['test_auc'] >= threshold].iloc[0]
print(f"\n Elbow point: {elbow['n_features']} features")
print(f"   Test AUC:    {elbow['test_auc']}")
print(f"   Gini:        {elbow['gini']}")


# ── Step 7: Extract final features ──
final_n = elbow['n_features']
shap_selected_features = ranked_features[:final_n]

print(f"\nFinal SHAP selected features ({final_n}):")
print(shap_selected_features)

# ── Step 8: Flag in dataframe for voting matrix ──
shap_df['SHAP_SELECTED'] = shap_df['feature'].isin(shap_selected_features).astype(int)

c:\ProgramData\anaconda3\envs\homecredit\Lib\site-packages\shap\explainers\_linear.py:123: FutureWarning: The feature_perturbation option is now deprecated in favor of using the appropriate masker (maskers.Independent, maskers.Partition or maskers.Impute).
  warnings.warn(wmsg, FutureWarning)


Estimating transforms:   0%|          | 0/1000 [00:00<?, ?it/s]

SHAP Feature Ranking:
                             feature  shap_importance
0                EXT_SOURCE_WEIGHTED         0.381384
1                       EXT_SOURCE_3         0.160209
2                       EXT_SOURCE_1         0.152634
3               ANNUITY_CREDIT_RATIO         0.144643
4                     YEARS_EMPLOYED         0.129477
5                NAME_EDUCATION_TYPE         0.126935
6                        CODE_GENDER         0.115723
7                    AMT_GOODS_PRICE         0.108526
8                 GOODS_CREDIT_RATIO         0.108465
9                   OCCUPATION_GROUP         0.105607
10            PA_AVG_AMT_ANNUITY_POS         0.102048
11                  EXT_SOURCE_RANGE         0.097335
12                         ORG_GROUP         0.093087
13             CREDIT_GOODS_DIFF_AMT         0.093009
14      IP_RATIO_LATE_PAYMENTS_2160D         0.090976
15     IP_RATIO_EARLY_PAYMENTS_1080D         0.090929
16            B_DEBT_TO_CREDIT_RATIO         0.088855
17    

In [46]:
shap_best_featuers = shap_df[shap_df['SHAP_SELECTED']==1]['feature'].tolist()
shap_removed_featuers = shap_df[shap_df['SHAP_SELECTED']==0]['feature'].tolist()

In [47]:
# SELECTEF THE C VALUE AND RETRAIN
final_model = LogisticRegression(
    penalty='l2', solver='lbfgs', C=1.0,
    class_weight='balanced', max_iter=2000, random_state=42
)

final_model.fit(X_train[shap_best_featuers],y_train)
train_prob = final_model.predict_proba(X_train[shap_best_featuers])
train_prob = train_prob[:,1]

test_prob = final_model.predict_proba( X_test[shap_best_featuers])
test_prob = test_prob[:,1]


c:\ProgramData\anaconda3\envs\homecredit\Lib\site-packages\sklearn\utils\validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [48]:
auc_gini_ks(y_test,test_prob)

AUC 0.7665449080633308
Gini 0.5330898161266615
KS 0.40787958669403857


In [49]:
auc_gini_ks(y_train,train_prob)

AUC 0.7669888150940589
Gini 0.5339776301881178
KS 0.40172746092306727


## VOTING  MATRIX FROM ALL THOSE 6 techniques

In [50]:
feature_df_1 = importance[['feature','LOG_REG_TOP_30']]
feature_df_2 = importance_1[['feature','LOG_REG_CORR_TOP_30']]
feature_df_3  = importance_2[['feature','RFE_TOP_30']]
feature_df_4 = importance_3[['feature','LASSO_SELECTED_FEATURES']]
feature_df_5 = importance_4[['feature','STABILITY_SELECTED']]
feature_df_6 = perm_df[['feature','PERM_BEST_SELECTED']]
feature_df_7 = shap_df[['feature','SHAP_SELECTED']] 

In [51]:
from functools import reduce

# Merge all 7 methods on feature
dfs = [feature_df_1, feature_df_2, feature_df_3, 
       feature_df_4, feature_df_5, feature_df_6, feature_df_7]

voting_matrix = reduce(lambda left, right: 
                       pd.merge(left, right, on='feature', how='outer'), dfs)

# Fill NaN with 0 (feature not selected by that method)
voting_matrix = voting_matrix.fillna(0)


In [52]:
cols = ['LOG_REG_TOP_30', 'LOG_REG_CORR_TOP_30', 'RFE_TOP_30',
               'LASSO_SELECTED_FEATURES', 'STABILITY_SELECTED',
               'PERM_BEST_SELECTED', 'SHAP_SELECTED']

voting_matrix['vote_score'] = voting_matrix[cols].sum(axis=1)

In [53]:
# Merge IV scores
voting_matrix = voting_matrix.merge(iv_df[['feature','IV']], on='feature', how='left')

# Sort by vote score first, then IV as tiebreaker
voting_matrix = voting_matrix.sort_values(
    ['vote_score', 'IV'], ascending=False
).reset_index(drop=True)

In [54]:
voting_matrix.head()

,feature,LOG_REG_TOP_30,LOG_REG_CORR_TOP_30,RFE_TOP_30,LASSO_SELECTED_FEATURES,STABILITY_SELECTED,PERM_BEST_SELECTED,SHAP_SELECTED,vote_score,IV
0,EXT_SOURCE_WEIGHTED,1,1.0,1.0,1.0,1.0,1.0,1.0,7.0,0.626102
1,CREDIT_EXT_SOURCE_PRODUCT,1,1.0,1.0,1.0,1.0,1.0,1.0,7.0,0.183983
2,ANNUITY_CREDIT_RATIO,1,1.0,1.0,1.0,1.0,1.0,1.0,7.0,0.145948
3,YEARS_EMPLOYED,1,1.0,1.0,1.0,1.0,1.0,1.0,7.0,0.114435
4,AMT_GOODS_PRICE,1,1.0,1.0,1.0,1.0,1.0,1.0,7.0,0.102669


## 7) VOTING top 25 fetures

* select the top 20 voting feature and again retrain the log reg model

In [55]:
top_25 = voting_matrix.head(25)['feature'].tolist()

In [56]:
# SELECTEF THE C VALUE AND RETRAIN
final_model = LogisticRegression(
        penalty='l2',          # removes weak features
        solver='liblinear',    # required for L1
        C=1,                 # strong regularization
        class_weight='balanced',
        max_iter=2000,
        random_state=42
    )

final_model.fit(X_train[top_25],y_train.values.ravel())
train_prob = final_model.predict_proba(X_train[top_25])
train_prob = train_prob[:,1]

test_prob = final_model.predict_proba(X_test[top_25])
test_prob = test_prob[:,1]


In [57]:
auc_gini_ks(y_train,train_prob)

AUC 0.7666754024461269
Gini 0.5333508048922537
KS 0.4026925624960802


In [58]:
auc_gini_ks(y_test,test_prob)

AUC 0.7667798898063086
Gini 0.5335597796126172
KS 0.40853773293025714


## 7) VOTING top 20 fetures

* select the top 20 voting feature and again retrain the log reg model

In [59]:
top_20 = voting_matrix.head(20)['feature'].tolist()

In [ ]:
# SELECTEF THE C VALUE AND RETRAIN
final_model = LogisticRegression(
        penalty='l2',          # removes weak features
        solver='liblinear',    # required for L1
        C=1,                 # strong regularization
        class_weight='balanced',
        max_iter=2000,
        random_state=42
    )

final_model.fit(X_train[top_20],y_train.values.ravel())
train_prob = final_model.predict_proba(X_train[top_20])
train_prob = train_prob[:,1]

test_prob = final_model.predict_proba(X_test[top_20])
test_prob = test_prob[:,1]

In [61]:
auc_gini_ks(y_train,train_prob)

AUC 0.7618137277638732
Gini 0.5236274555277465
KS 0.3938389489822642


In [62]:
auc_gini_ks(y_test,test_prob)

AUC 0.7616379355335238
Gini 0.5232758710670475
KS 0.3929192985088552


# FINAL FEATURE SELECTION

* top 20 feature are giving the best auc gini and ks!
* stepwise log reg with 67 features:  0.7736
* log reg l2 : 20 feature : 0.7618
* 0.011799 loss of auc trade for 47 featuresss and that is great!!! 
* 0.0240 loss gini
* we can try other combinations of feature with those top 20 can it improve the auc or not

* CREDIT_EXT_SOURCE_PRODUCT removed due to domian problem

In [142]:
top_25_set   = set(top_25)
shap_set  = set(shap_selected_features)
voting_set = set(top_20)

# Features in BOTH rfe and shap but NOT in voting 20
# These are high confidence additions
both_but_not_voting = (top_25_set & shap_set) - voting_set

print(f"High confidence additions: {len(both_but_not_voting)}")
print(both_but_not_voting)

High confidence additions: 4
{'IP_NUM_COMPLETED_LOANS', 'PA_AVG_RISK_WEIGHT_1080D', 'OCCUPATION_GROUP', 'B_CREDIT_DURATION_MIN'}


In [143]:
top_20_copy = top_20.copy()

In [144]:
# SELECTEF THE C VALUE AND RETRAIN
final_model = LogisticRegression(
        penalty='l2',          # removes weak features
        solver='liblinear',    # required for L1
        C=1,                 # strong regularization
        class_weight='balanced',
        max_iter=2000,
        random_state=42
    )
top_20_copy = top_20_copy + list(both_but_not_voting)


final_model.fit(X_train[top_20_copy],y_train.values.ravel())
train_prob = final_model.predict_proba(X_train[top_20_copy])
train_prob = train_prob[:,1]

test_prob = final_model.predict_proba(X_test[top_20_copy])
test_prob = test_prob[:,1]

In [145]:
auc_gini_ks(y_train,train_prob)

AUC 0.7644890175299117
Gini 0.5289780350598234
KS 0.3996189254170566


* 24 feature till now the 

In [146]:
cb_features = []

for feature in corr_selected_features:
    if feature.startswith('CB'):
        cb_features.append(feature)

In [147]:
cb_features

['CB_AVG_ATM_WITHDRAWAL_FREQ_6M', 'CB_WT_CREDIT_UTIL_TREND_3M_12M']

* cb features are removed by diff methods so i added manually the signal of that was purely missing so

In [150]:
# SELECTEF THE C VALUE AND RETRAIN
final_model = LogisticRegression(
        penalty='l2',          # removes weak features
        solver='liblinear',    # required for L1
        C=1,                 # strong regularization
        class_weight='balanced',
        max_iter=2000,
        random_state=42
    )
final_features = top_20_copy + cb_features


final_model.fit(X_train[final_features],y_train.values.ravel())
train_prob = final_model.predict_proba(X_train[final_features])
train_prob = train_prob[:,1]

test_prob = final_model.predict_proba(X_test[final_features])
test_prob = test_prob[:,1]

In [151]:
auc_gini_ks(y_train,train_prob)

AUC 0.7669379480763527
Gini 0.5338758961527055
KS 0.4033771029393526


##### DONE

In [1]:
final_selected_features = ['EXT_SOURCE_WEIGHTED',
'CREDIT_EXT_SOURCE_PRODUCT','ANNUITY_CREDIT_RATIO','YEARS_EMPLOYED','AMT_GOODS_PRICE','IP_WORST_DPD_720D','GOODS_CREDIT_RATIO','IP_RATIO_LATE_PAYMENTS_2160D',
'B_NUM_ACTIVE_CREDIT_720D','ORG_GROUP','REGION_RATING_CLIENT_W_CITY','NAME_EDUCATION_TYPE','PA_RATIO_CREDIT_APPLICATION_Cash','PA_RATIO_CREDIT_APPLICATION_POS','CODE_GENDER',
'PA_AVG_AMT_ANNUITY_POS','EXT_SOURCE_3','EXT_SOURCE_1','B_DEBT_TO_CREDIT_RATIO','PA_RATIO_APPROVED_LOANS','IP_NUM_COMPLETED_LOANS','PA_AVG_RISK_WEIGHT_1080D','OCCUPATION_GROUP',
'B_CREDIT_DURATION_MIN','CB_AVG_ATM_WITHDRAWAL_FREQ_6M','CB_WT_CREDIT_UTIL_TREND_3M_12M'
]

In [2]:
from pathlib import Path
from src.entity.artifact_entity import FeatureBinMergingArtifact
import json

artifact = FeatureBinMergingArtifact()
final_dir = artifact.final_model_features

final_dir.parent.mkdir(exist_ok=True,parents=True)
with open(final_dir, "w") as f:
    json.dump(final_selected_features, f, indent=4)
print(final_dir)

D:\home loan credit risk\artifact\data\final\model_features.json
